# Inference Benchmark: CPU 기반 Whisper 추론 속도 비교

**목적**: GPU 없는 환경에서의 추론 가능성을 검토하기 위해 CPU 기반 Whisper 추론을 벤치마크합니다.

**내용**: `elmenwol/whisper-small_aihub_child` 모델을 CPU에서 로드하고 4가지 추론 방식의 속도 및 메모리를 비교합니다.

**최적화 기법 비교**:
| 기법 | 설명 |
| --- | --- |
| **Direct Inference (FP32)** | HuggingFace processor + model.generate() 직접 호출 (baseline) |
| **Dynamic Quantization (INT8)** | torch.quantization.quantize_dynamic으로 Linear 레이어 INT8 변환 |
| **ONNX Runtime** | ONNX 포맷 변환 후 onnxruntime으로 추론 |
| **멀티스레드 비교** | torch.set_num_threads로 1/2/4/8 스레드별 성능 비교 |

---

# CPU Inference 분석

- **환경**: Google Colab T4 GPU (15GB VRAM) · `elmenwol/whisper-small_aihub_child`

---

## 단건 추론 속도

| 방법 | 시간 | vs CPU Direct | vs CPU Best | 메모리 |
|------|------|:---:|:---:|------|
| **CPU** Direct (FP32) | 16.70s | 1.00x | — | 2,516 MB (RSS) |
| **CPU** Dynamic Quant (INT8) | 65.16s | 0.26x | — | 3,693 MB (RSS) |
| **CPU** ONNX Runtime | 12.75s | 1.31x | 1.00x | 5,834 MB (RSS) |

---

## 처리량 (N=50)

| 환경 | 방법 | QPS | 50건 총시간 | P95 지연 |
|------|------|:---:|:---:|:---:|
| CPU | Direct (순차) | ~0.06 | ~835s (추정) | ~16.7s |
| CPU | ONNX (순차) | ~0.08 | ~637s (추정) | ~12.7s |

- CPU는 순차 처리만 가능하므로 50건 ≈ 50×12.7s = 635초.

---

## 메모리 효율

- Direct: 2,516 MB RAM
- INT8: 3,693 MB RAM
- ONNX: 5,834 MB RAM

- CPU 최소 메모리 (FP32): **2,516 MB**

---

## CPU 최적화의 한계

| CPU 기법 | 효과 | 문제점 |
|----------|------|--------|
| Dynamic Quantization (INT8) | **0.26x (역효과)** | Whisper 디코더의 attention이 INT8에 비적합, 오히려 4배 느림 |
| ONNX Runtime | **1.31x** | 속도 개선 미미, 메모리 2.3배 증가 (5,834MB) |


## 0. 환경 설정 및 의존성

In [2]:
# 필요 시 설치
# !pip install -q transformers torch torchaudio optimum[onnxruntime] psutil

zsh:1: no matches found: optimum[onnxruntime]


In [9]:
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 4.0 MB/s  0:00:00 eta 0:00:01


In [3]:
import os
import time
import gc
import statistics

import torch
import torchaudio
import psutil
import numpy as np

from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import logging as transformers_log

transformers_log.set_verbosity_error()

print(f"PyTorch version: {torch.__version__}")
print(f"CPU: {os.cpu_count()} cores")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"Current torch threads: {torch.get_num_threads()}")

/opt/anaconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0
CPU: 8 cores
RAM: 8.0 GB
Current torch threads: 4


In [4]:
# ===== 설정값 =====
MODEL_NAME = "elmenwol/whisper-small_aihub_child"
AUDIO_FILE = "/Users/jaeho/Workspace/LetsPlay-ai-research/experiments/05_tts_experiments/voice_sample_jh1.wav"  # 테스트 오디오 파일 경로 (환경에 맞게 수정)
NUM_RUNS = 5  # 반복 측정 횟수
WARMUP_RUNS = 1  # 웜업 횟수
SAMPLING_RATE = 16000

# 결과 저장용
benchmark_results = {}


def get_memory_mb():
    """현재 프로세스 RSS 메모리 (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2


def benchmark_inference(name, inference_fn, num_runs=NUM_RUNS, warmup=WARMUP_RUNS):
    """
    추론 함수를 반복 실행하여 시간을 측정합니다.

    Args:
        name: 벤치마크 이름
        inference_fn: 인자 없이 호출 가능한 추론 함수 (텍스트 반환)
        num_runs: 측정 반복 횟수
        warmup: 웜업 횟수 (측정에 포함되지 않음)

    Returns:
        dict: {times, mean, std, text}
    """
    # 웜업
    for _ in range(warmup):
        _ = inference_fn()

    gc.collect()
    mem_before = get_memory_mb()

    times = []
    result_text = ""
    for i in range(num_runs):
        start = time.perf_counter()
        result_text = inference_fn()
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    mem_after = get_memory_mb()

    mean_time = statistics.mean(times)
    std_time = statistics.stdev(times) if len(times) > 1 else 0.0

    result = {
        "times": times,
        "mean": mean_time,
        "std": std_time,
        "mem_before": mem_before,
        "mem_after": mem_after,
        "text": result_text,
    }

    print(f"[{name}]")
    print(f"  결과: {result_text}")
    print(f"  평균 시간: {mean_time:.4f}s ± {std_time:.4f}s")
    print(f"  각 실행: {[f'{t:.4f}s' for t in times]}")
    print(f"  메모리 (RSS): {mem_before:.1f} → {mem_after:.1f} MB")

    benchmark_results[name] = result
    return result


def load_audio(audio_path, sr=SAMPLING_RATE):
    """오디오 파일 로드 및 리샘플링"""
    waveform, sample_rate = torchaudio.load(audio_path)
    if sample_rate != sr:
        waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=sr)(waveform)
    return waveform[0].numpy()  # mono, numpy array


print(f"테스트 오디오: {AUDIO_FILE}")
print(f"반복 측정 횟수: {NUM_RUNS}회 (+ 웜업 {WARMUP_RUNS}회)")

테스트 오디오: /Users/jaeho/Workspace/LetsPlay-ai-research/experiments/05_tts_experiments/voice_sample_jh1.wav
반복 측정 횟수: 5회 (+ 웜업 1회)


---
## 1. Direct Inference: HuggingFace FP32 (Baseline)

`WhisperProcessor`와 `model.generate()`를 CPU에서 FP32로 직접 호출합니다.
이것이 기본 성능 기준선(baseline)이 됩니다.

In [5]:
# 모델 & Processor 로드
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model_fp32 = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
model_fp32 = model_fp32.to("cpu")
model_fp32.eval()

# 모델 메모리 크기 (파라미터 기준)
param_size_mb = sum(p.nelement() * p.element_size() for p in model_fp32.parameters()) / 1024**2
print(f"모델 파라미터 크기: {param_size_mb:.1f} MB")
print(f"모델 로드 후 RSS: {get_memory_mb():.1f} MB")

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 5604.04it/s]


모델 파라미터 크기: 922.1 MB
모델 로드 후 RSS: 193.3 MB


In [6]:
def transcribe_direct(audio_file, model, processor, device="cpu"):
    """processor + model.generate()로 직접 추론"""
    audio_input = load_audio(audio_file)

    input_features = processor(
        audio_input,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
    ).input_features.to(device=device, dtype=torch.float32)

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="ko",
        task="transcribe",
    )

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=255,
        )

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True,
    )[0]
    return transcription

In [7]:
# Baseline 벤치마크
torch.set_num_threads(4)  # 기본 스레드 수 고정
print(f"스레드 수: {torch.get_num_threads()}")

benchmark_inference(
    "1. Direct Inference (FP32)",
    lambda: transcribe_direct(AUDIO_FILE, model_fp32, processor),
)

스레드 수: 4


objc[68008]: Class AVFFrameReceiver is implemented in both /opt/anaconda3/envs/torch/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x1114f03a8) and /opt/homebrew/Cellar/ffmpeg/7.1.1_3/lib/libavdevice.61.3.100.dylib (0x14ec94328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[68008]: Class AVFAudioReceiver is implemented in both /opt/anaconda3/envs/torch/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x1114f03f8) and /opt/homebrew/Cellar/ffmpeg/7.1.1_3/lib/libavdevice.61.3.100.dylib (0x14ec94378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


[1. Direct Inference (FP32)]
  결과: 그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.
  평균 시간: 3.4850s ± 0.7440s
  각 실행: ['4.8041s', '3.3142s', '3.1205s', '3.0445s', '3.1416s']
  메모리 (RSS): 1004.5 → 1202.7 MB


{'times': [4.804141542001162,
  3.3142145419988083,
  3.1204888749925885,
  3.0445338749996154,
  3.141572874999838],
 'mean': 3.4849903417984023,
 'std': 0.7439964781073186,
 'mem_before': 1004.453125,
 'mem_after': 1202.6875,
 'text': '그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.'}

---
## 2. Dynamic Quantization (INT8)

`torch.quantization.quantize_dynamic`으로 Linear 레이어를 INT8로 변환합니다.
모델 크기가 줄고 CPU 연산이 빨라지며, 정확도 손실은 미미합니다.

In [8]:
torch.backends.quantized.engine = 'qnnpack'

# INT8 동적 양자화 적용
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},
    dtype=torch.qint8,
)

# 양자화된 모델 크기 추정
int8_param_size = 0
for name, param in model_int8.named_parameters():
    int8_param_size += param.nelement() * param.element_size()
int8_param_size_mb = int8_param_size / 1024**2

print(f"INT8 모델 파라미터 크기: {int8_param_size_mb:.1f} MB (FP32: {param_size_mb:.1f} MB)")
print(f"모델 로드 후 RSS: {get_memory_mb():.1f} MB")

/var/folders/2f/8pgmnc8j1hvcgz5q9kcrwvsh0000gn/T/ipykernel_68008/3700968111.py:4: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


INT8 모델 파라미터 크기: 165.5 MB (FP32: 922.1 MB)
모델 로드 후 RSS: 2011.9 MB


In [9]:
# INT8 벤치마크
torch.set_num_threads(4)
print(f"스레드 수: {torch.get_num_threads()}")

benchmark_inference(
    "2. Dynamic Quantization (INT8)",
    lambda: transcribe_direct(AUDIO_FILE, model_int8, processor),
)

스레드 수: 4


[W306 14:13:42.988358000 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


[2. Dynamic Quantization (INT8)]
  결과: 그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.
  평균 시간: 2.9608s ± 0.0441s
  각 실행: ['2.9529s', '2.9314s', '2.9493s', '3.0379s', '2.9328s']
  메모리 (RSS): 1158.0 → 1334.2 MB


{'times': [2.952894166999613,
  2.931364207994193,
  2.949250208999729,
  3.0379051249910844,
  2.9328028339950833],
 'mean': 2.9608433085959405,
 'std': 0.044134447840134754,
 'mem_before': 1157.96875,
 'mem_after': 1334.234375,
 'text': '그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.'}

---
## 3. ONNX Runtime 추론

HuggingFace `optimum` 라이브러리로 모델을 ONNX 포맷으로 변환한 뒤,
`onnxruntime`으로 추론합니다. CPU 환경에서 대폭적인 속도 향상이 가능합니다.

In [10]:
try:
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
    ONNX_AVAILABLE = True
except ImportError:
    print("optimum[onnxruntime] 미설치. 이 섹션을 건너뜁니다.")
    print("설치: pip install optimum[onnxruntime]")
    ONNX_AVAILABLE = False

if ONNX_AVAILABLE:
    ONNX_EXPORT_DIR = "./whisper-small-aihub-onnx"

    # ONNX 모델 로드 (없으면 자동 변환)
    model_onnx = ORTModelForSpeechSeq2Seq.from_pretrained(
        MODEL_NAME,
        export=True,  # HuggingFace → ONNX 자동 변환
        provider="CPUExecutionProvider",
    )

    # 변환된 ONNX 모델 로컬 저장 (재사용)
    model_onnx.save_pretrained(ONNX_EXPORT_DIR)
    processor.save_pretrained(ONNX_EXPORT_DIR)
    print(f"ONNX 모델 저장 완료: {ONNX_EXPORT_DIR}")
    print(f"모델 로드 후 RSS: {get_memory_mb():.1f} MB")

optimum[onnxruntime] 미설치. 이 섹션을 건너뜁니다.
설치: pip install optimum[onnxruntime]


In [11]:
def transcribe_onnx(audio_file, model, processor):
    """ONNX Runtime으로 추론"""
    audio_input = load_audio(audio_file)

    input_features = processor(
        audio_input,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
    ).input_features

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="ko",
        task="transcribe",
    )

    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids,
        max_new_tokens=255,
    )

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True,
    )[0]
    return transcription


if ONNX_AVAILABLE:
    benchmark_inference(
        "3. ONNX Runtime",
        lambda: transcribe_onnx(AUDIO_FILE, model_onnx, processor),
    )

---
## 4. 멀티스레드 비교 (`torch.set_num_threads`)

CPU 추론에서 PyTorch 스레드 수에 따른 속도 변화를 측정합니다.
FP32 baseline 모델로 1, 2, 4, 8 스레드를 비교합니다.

In [12]:
thread_counts = [1, 2, 4, 8]
thread_results = {}

for n_threads in thread_counts:
    torch.set_num_threads(n_threads)
    label = f"4. FP32 ({n_threads} threads)"

    result = benchmark_inference(
        label,
        lambda: transcribe_direct(AUDIO_FILE, model_fp32, processor),
        num_runs=3,  # 스레드 테스트는 3회로 축소
        warmup=1,
    )
    thread_results[n_threads] = result["mean"]
    print()

# 스레드 수 원복
torch.set_num_threads(4)

[4. FP32 (1 threads)]
  결과: 그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.
  평균 시간: 3.1623s ± 0.0682s
  각 실행: ['3.1726s', '3.0895s', '3.2246s']
  메모리 (RSS): 1462.9 → 1374.5 MB

[4. FP32 (2 threads)]
  결과: 그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.
  평균 시간: 3.0412s ± 0.0306s
  각 실행: ['3.0098s', '3.0431s', '3.0708s']
  메모리 (RSS): 1511.2 → 1472.5 MB

[4. FP32 (4 threads)]
  결과: 그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.
  평균 시간: 2.9445s ± 0.0882s
  각 실행: ['2.8921s', '3.0463s', '2.8950s']
  메모리 (RSS): 1535.0 → 1492.1 MB

[4. FP32 (8 threads)]
  결과: 그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.
  평균 시간: 2.8441s ± 0.0607s
  각 실행: ['2.8122s', '2.9141s', '2.8061s']
  메모리 (RSS): 1528.9 → 1356.0 MB



---
## 5. 벤치마크 결과 비교

In [13]:
# --- 최적화 기법 비교 (스레드 4 고정, 1~3번 기법 비교) ---
main_methods = [
    "1. Direct Inference (FP32)",
    "2. Dynamic Quantization (INT8)",
]
if ONNX_AVAILABLE:
    main_methods.append("3. ONNX Runtime")

print("=" * 85)
print(f"{'Method':<35} {'Mean':>10} {'Std':>10} {'Speedup':>10} {'RSS(MB)':>10}")
print("-" * 85)

baseline_time = benchmark_results[main_methods[0]]["mean"]

for method in main_methods:
    data = benchmark_results[method]
    speedup = baseline_time / data["mean"] if data["mean"] > 0 else float("inf")
    print(
        f"{method:<35} {data['mean']:>9.4f}s {data['std']:>9.4f}s "
        f"{speedup:>9.2f}x {data['mem_after']:>9.1f}"
    )

print("=" * 85)

# 최고 성능 요약
fastest = min(main_methods, key=lambda k: benchmark_results[k]["mean"])
print(f"\n⚡ 최고 속도: {fastest}")
print(f"   → {benchmark_results[fastest]['mean']:.4f}s ({baseline_time / benchmark_results[fastest]['mean']:.1f}x faster)")

Method                                    Mean        Std    Speedup    RSS(MB)
-------------------------------------------------------------------------------------
1. Direct Inference (FP32)             3.4850s    0.7440s      1.00x    1202.7
2. Dynamic Quantization (INT8)         2.9608s    0.0441s      1.18x    1334.2

⚡ 최고 속도: 2. Dynamic Quantization (INT8)
   → 2.9608s (1.2x faster)


In [14]:
# --- 멀티스레드 비교 ---
print("\n" + "=" * 55)
print(f"{'Threads':>10} {'Mean Time':>12} {'Speedup':>10}")
print("-" * 55)

baseline_thread_time = thread_results.get(1, list(thread_results.values())[0])

for n_threads, mean_time in thread_results.items():
    speedup = baseline_thread_time / mean_time if mean_time > 0 else float("inf")
    print(f"{n_threads:>10} {mean_time:>11.4f}s {speedup:>9.2f}x")

print("=" * 55)

optimal_threads = min(thread_results, key=thread_results.get)
print(f"\n🧵 최적 스레드 수: {optimal_threads}")
print(f"   → {thread_results[optimal_threads]:.4f}s ({baseline_thread_time / thread_results[optimal_threads]:.1f}x vs 1 thread)")


   Threads    Mean Time    Speedup
-------------------------------------------------------
         1      3.1623s      1.00x
         2      3.0412s      1.04x
         4      2.9445s      1.07x
         8      2.8441s      1.11x

🧵 최적 스레드 수: 8
   → 2.8441s (1.1x vs 1 thread)


In [15]:
# --- 전사 결과 정확성 비교 ---
print("\n" + "=" * 70)
print("전사 결과 비교 (동일 오디오)")
print("-" * 70)

for method in main_methods:
    text = benchmark_results[method]["text"]
    print(f"\n[{method}]")
    print(f"  {text}")

print("\n" + "=" * 70)


전사 결과 비교 (동일 오디오)
----------------------------------------------------------------------

[1. Direct Inference (FP32)]
  그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.

[2. Dynamic Quantization (INT8)]
  그중 하나는 에너지 주니가 낮은 결합성 궤도함수이고, 다른 하나는 에너지 주니가 높은 반결합성 궤도함수이며, 또 다른 하나는 중간 에너지 주니를 가진 비결합성 궤도함수이다.



In [16]:
# 메모리 정리
del model_fp32, model_int8
if ONNX_AVAILABLE:
    del model_onnx
gc.collect()
print(f"정리 후 RSS: {get_memory_mb():.1f} MB")

정리 후 RSS: 573.6 MB
